In [ ]:
!pip install openmeteo_requests
!pip install requests_cache
!pip install retry_requests
!pip install openpyxl

In [7]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time

# ── Setup session avec cache 1h et retry automatique ──────────────
cache_session = requests_cache.CachedSession('.cache_meteo', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# ── Départements côtiers français ────────────────────────────────
DEPARTEMENTS_COTIERS = {
    # Manche & Mer du Nord
    "Nord (59)":            (51.03, 2.38,  "Manche"),
    "Pas-de-Calais (62)":   (50.72, 1.61,  "Manche"),
    "Somme (80)":           (50.19, 1.62,  "Manche"),
    "Seine-Maritime (76)":  (49.74, 0.41,  "Manche"),
    "Calvados (14)":        (49.18, -0.37, "Manche"),
    "Manche (50)":          (49.12, -1.41, "Manche"),
    "Ille-et-Vilaine (35)": (48.64, -1.97, "Manche"),
    # Atlantique
    "Côtes-d'Armor (22)":   (48.51, -2.76, "Atlantique"),
    "Finistère (29)":       (48.39, -4.49, "Atlantique"),
    "Morbihan (56)":        (47.66, -2.76, "Atlantique"),
    "Loire-Atlantique (44)":(47.28, -2.21, "Atlantique"),
    "Vendée (85)":          (46.67, -1.94, "Atlantique"),
    "Charente-Maritime (17)":(45.75,-1.05, "Atlantique"),
    "Gironde (33)":         (45.15, -1.05, "Atlantique"),
    "Landes (40)":          (43.95, -1.40, "Atlantique"),
    "Pyrénées-Atlantiques (64)":(43.49,-1.56,"Atlantique"),
    # Méditerranée
    "Pyrénées-Orientales (66)":(42.70,3.02, "Méditerranée"),
    "Hérault (34)":         (43.44, 3.70,  "Méditerranée"),
    "Gard (30)":            (43.56, 4.19,  "Méditerranée"),
    "Bouches-du-Rhône (13)":(43.29, 5.38,  "Méditerranée"),
    "Var (83)":             (43.12, 6.17,  "Méditerranée"),
    "Alpes-Maritimes (06)": (43.71, 7.27,  "Méditerranée"),
    "Haute-Corse (2B)":     (42.70, 9.45,  "Méditerranée"),
    "Corse-du-Sud (2A)":    (41.92, 8.74,  "Méditerranée"),
}

# ── Variables météo à récupérer ───────────────────────────────────
VARIABLES_HORAIRES = [
    "temperature_2m",          # Température air (°C)
    "relative_humidity_2m",     # Humidité relative (%)
    "apparent_temperature",     # Température ressentie (°C)
    "precipitation",            # Précipitations (mm)
    "wind_speed_10m",           # Vitesse vent à 10m (km/h)
    "wind_gusts_10m",           # Rafales (km/h)
    "wind_direction_10m",       # Direction vent (°)
    "weather_code",             # Code météo WMO
    "cloud_cover",              # Couverture nuageuse (%)
    "pressure_msl",             # Pression mer (hPa)
    "shortwave_radiation",      # Rayonnement solaire (W/m²)
    "visibility",               # Visibilité (m) — important côtier
]

VARIABLES_JOURNALIERES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "wind_direction_10m_dominant",
    "sunrise",
    "sunset",
    "uv_index_max",
    "weather_code",
]

# ── Récupération des données ──────────────────────────────────────
def fetch_meteo_departement(nom, lat, lon, facade):
    params = {
        "latitude":          lat,
        "longitude":         lon,
        "hourly":            VARIABLES_HORAIRES,
        "daily":             VARIABLES_JOURNALIERES,
        "current": ["temperature_2m", "wind_speed_10m",
                     "wind_gusts_10m", "precipitation",
                     "weather_code", "pressure_msl"],
        "models":            "meteofrance_seamless",
        "timezone":          "Europe/Paris",
        "forecast_days":     4,
        "wind_speed_unit":   "kmh",
        "temperature_unit":  "celsius",
        "precipitation_unit":"mm",
    }
    url = "https://api.open-meteo.com/v1/meteofrance"
    responses = openmeteo.weather_api(url, params=params)
    r = responses[0]

    # Données horaires
    h = r.Hourly()
    times = pd.date_range(
        start=pd.to_datetime(h.Time(), unit="s", utc=True),
        end=pd.to_datetime(h.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=h.Interval()),
        inclusive="left"
    ).tz_convert("Europe/Paris")

    df_h = pd.DataFrame({"time": times})
    for i, var in enumerate(VARIABLES_HORAIRES):
        df_h[var] = h.Variables(i).ValuesAsNumpy()
    df_h["departement"] = nom
    df_h["facade"]      = facade
    df_h["latitude"]    = lat
    df_h["longitude"]   = lon

    return df_h

# ── Boucle sur tous les départements ─────────────────────────────
all_dfs = []
for nom, (lat, lon, facade) in DEPARTEMENTS_COTIERS.items():
    print(f"Récupération : {nom} ({facade})...")
    df = fetch_meteo_departement(nom, lat, lon, facade)
    all_dfs.append(df)
    time.sleep(0.1)  # Respect rate limit

df_cotes = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ {len(df_cotes)} lignes — {df_cotes['departement'].nunique()} départements")

Récupération : Nord (59) (Manche)...
Récupération : Pas-de-Calais (62) (Manche)...
Récupération : Somme (80) (Manche)...
Récupération : Seine-Maritime (76) (Manche)...
Récupération : Calvados (14) (Manche)...
Récupération : Manche (50) (Manche)...
Récupération : Ille-et-Vilaine (35) (Manche)...
Récupération : Côtes-d'Armor (22) (Atlantique)...
Récupération : Finistère (29) (Atlantique)...
Récupération : Morbihan (56) (Atlantique)...
Récupération : Loire-Atlantique (44) (Atlantique)...
Récupération : Vendée (85) (Atlantique)...
Récupération : Charente-Maritime (17) (Atlantique)...
Récupération : Gironde (33) (Atlantique)...
Récupération : Landes (40) (Atlantique)...
Récupération : Pyrénées-Atlantiques (64) (Atlantique)...
Récupération : Pyrénées-Orientales (66) (Méditerranée)...
Récupération : Hérault (34) (Méditerranée)...
Récupération : Gard (30) (Méditerranée)...
Récupération : Bouches-du-Rhône (13) (Méditerranée)...
Récupération : Var (83) (Méditerranée)...
Récupération : Alpes-Mari

In [12]:
# ── Export ───────────────────────────────────────────────────────

# Supprime la timezone pour compatibilité Excel
df_export = df_cotes.copy()
df_export["time"] = df_export["time"].dt.tz_localize(None)

# ── CSV ──────────────────────────────────────────────────────────
df_export.to_csv("meteo_cotes_france.csv", index=False, encoding="utf-8-sig")
print("✓ CSV exporté : meteo_cotes_france.csv")

# ── Conditions actuelles par département ─────────────────────────
now_df = df_export[df_export["time"] == df_export.groupby("departement")["time"].transform("min")]
print("\n── Conditions actuelles ──")
print(now_df[["departement", "facade", "temperature_2m",
              "wind_speed_10m", "wind_gusts_10m", "precipitation"]].to_string(index=False))

# ── Vent max par façade maritime ──────────────────────────────────
vent_max = (df_export
    .groupby("facade")["wind_gusts_10m"]
    .max()
    .sort_values(ascending=False)
    .reset_index())
vent_max.columns = ["Façade", "Rafales max (km/h)"]
print("\n── Rafales max par façade ──")
print(vent_max.to_string(index=False))

# ── Alerte pluie : départements > 5mm sur 4 jours ────────────────
pluie_alert = (df_export
    .groupby("departement")["precipitation"]
    .sum()
    .reset_index()
    .query("precipitation > 5")
    .sort_values("precipitation", ascending=False))
pluie_alert.columns = ["Département", "Précipitations totales (mm)"]
print("\n── Départements avec pluie > 5mm ──")
print(pluie_alert.to_string(index=False))

# ── Excel avec une feuille par façade ────────────────────────────
with pd.ExcelWriter("meteo_cotes_france.xlsx", engine="openpyxl") as writer:
    for facade, group in df_export.groupby("facade"):
        group.to_excel(writer, sheet_name=facade, index=False)
print("\n✓ Excel exporté : meteo_cotes_france.xlsx")

✓ CSV exporté : meteo_cotes_france.csv

── Conditions actuelles ──
              departement       facade  temperature_2m  wind_speed_10m  wind_gusts_10m  precipitation
                Nord (59)       Manche       12.974000       12.229406       24.119999            0.0
       Pas-de-Calais (62)       Manche       12.156500       11.525623       23.759998            0.0
               Somme (80)       Manche       11.004000       11.275530       13.320000            0.0
      Seine-Maritime (76)       Manche       11.187500        6.608722        8.640000            0.0
            Calvados (14)       Manche       13.450000       14.578890       22.680000            0.0
              Manche (50)       Manche       10.235001        8.557102        7.920000            0.0
     Ille-et-Vilaine (35)       Manche       11.487000       12.864649       21.959999            0.0
       Côtes-d'Armor (22)   Atlantique       11.580501        9.339208       21.959999            0.0
           Fini